<a href="https://colab.research.google.com/github/Kobby24/intro_to_AI_py/blob/main/GSE7390_BRCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import RFECV, mutual_info_classif
from sklearn.model_selection import StratifiedKFold, cross_val_score,train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix, roc_curve
)
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
import warnings
warnings.filterwarnings('ignore')
import io
import joblib



# LOAD DATA

In [2]:
metadata = pd.read_csv("/content/drive/MyDrive/MICRO_AI/GSE7390_BRCA_metadata.csv",index_col=0)
genes = pd.read_csv("/content/drive/MyDrive/MICRO_AI/GSE7390_BRCA_genes.csv",index_col=0)

In [3]:
metadata.head()

,sample_name,patient_id,file_name,hospital_source,age_at_diagnosis,tumor_size,surgery_type,histological_type,angioinvasion,lymphocytic_infiltration,...,risk_st_gallen,npi_score,risk_npi,aol_10yr_os_prob,risk_aol,risk_veridex,geo_title,geo_source_name,organism,geo_platform_id
Sample_ID,,,,,,,,,,,,,,,,,,,,,
GSM177885,VDXGUYU_4002,4002,10-4002_GUYA.CEL.gz,GUY,57,3.0,0,1.0,1.0,2.0,...,2.0,4.60,2.0,62.7,1,Poor,VDXGUYU_4002,VDXGUYU_4002,Homo sapiens,GPL96
GSM177886,VDXGUYU_4008,4008,26-4008_GUYA.CEL.gz,GUY,57,3.0,0,2.0,1.0,3.0,...,2.0,4.60,2.0,69.0,1,Poor,VDXGUYU_4008,VDXGUYU_4008,Homo sapiens,GPL96
GSM177887,VDXGUYU_4011,4011,6-4011_GUYA.CEL.gz,GUY,48,2.5,1,1.0,0.0,2.0,...,2.0,4.50,2.0,66.2,1,Poor,VDXGUYU_4011,VDXGUYU_4011,Homo sapiens,GPL96
GSM177888,VDXGUYU_4014,4014,9-4014_GUYA.CEL.gz,GUY,42,1.8,0,1.0,1.0,3.0,...,2.0,4.36,2.0,84.9,1,Poor,VDXGUYU_4014,VDXGUYU_4014,Homo sapiens,GPL96
GSM177889,VDXGUYU_4022,4022,219-4022_GUYA.CEL.gz,GUY,46,3.0,0,1.0,1.0,2.0,...,2.0,3.60,2.0,80.2,1,Poor,VDXGUYU_4022,VDXGUYU_4022,Homo sapiens,GPL96


In [4]:
genes.head()

,1007_s_at,1053_at,117_at,121_at,1255_g_at,1294_at,1316_at,1320_at,1405_i_at,1431_at,...,AFFX-r2-Hs28SrRNA-5_at,AFFX-r2-Hs28SrRNA-M_at,AFFX-r2-P1-cre-3_at,AFFX-r2-P1-cre-5_at,AFFX-ThrX-3_at,AFFX-ThrX-5_at,AFFX-ThrX-M_at,AFFX-TrpnX-3_at,AFFX-TrpnX-5_at,AFFX-TrpnX-M_at
GSM177885,10.780787,8.674201,7.738589,9.285511,6.610484,8.505162,6.867181,7.168175,6.252700,5.262454,...,8.921485,9.313626,14.888195,14.369094,2.929942,5.680176,1.753631,2.223181,4.013873,1.603252
GSM177886,11.335393,9.354759,7.763657,9.025809,4.809869,8.674612,5.097706,5.654260,9.892701,3.438981,...,8.696506,8.711194,15.011879,14.545059,2.667544,2.888221,1.767325,2.013092,5.355048,2.455589
GSM177887,11.028074,9.053889,6.327600,9.234409,4.621973,8.406754,6.932156,6.263502,8.777121,4.738016,...,7.975107,9.370440,15.170855,14.838521,3.258987,3.464008,4.875803,0.581058,4.054644,2.021403
GSM177888,11.847736,9.139895,7.032598,9.656416,5.589404,9.106449,7.751138,6.371654,11.993179,5.185123,...,9.130724,9.413045,15.126623,14.702279,3.865262,5.564328,3.361199,2.251075,4.760490,2.365204
GSM177889,12.359239,9.196708,7.873428,9.174188,5.325173,10.030778,7.156790,4.057771,6.359531,6.061211,...,8.028944,8.596815,15.492080,14.936844,3.321554,5.957565,2.400875,4.915962,6.120431,2.774229


In [5]:
expression = genes.loc["GSM177885"]

result = pd.DataFrame([expression.values])

result.insert(0, "sample", "GSM177885")

result.to_csv(
    "GSM177885_expression.csv",
    index=False,
    header=False
)

In [6]:
print(result.shape)

(1, 22284)


In [7]:
metadata.columns

Index(['sample_name', 'patient_id', 'file_name', 'hospital_source',
       'age_at_diagnosis', 'tumor_size', 'surgery_type', 'histological_type',
       'angioinvasion', 'lymphocytic_infiltration', 'lymph_node_status',
       'tumor_grade', 'estrogen_receptor_status', 'rfs_time', 'rfs_event',
       'os_time', 'os_event', 'dmfs_time', 'dmfs_event', 'tdm_time',
       'tdm_event', 'risk_st_gallen', 'npi_score', 'risk_npi',
       'aol_10yr_os_prob', 'risk_aol', 'risk_veridex', 'geo_title',
       'geo_source_name', 'organism', 'geo_platform_id'],
      dtype='object')

In [8]:
metadata.shape

(198, 31)

In [9]:
genes.shape

(198, 22283)

In [10]:
metadata.surgery_type.value_counts()

,count
surgery_type,
0,142
1,56


In [11]:
metadata.estrogen_receptor_status.value_counts()

,count
estrogen_receptor_status,
1,134
0,64


In [12]:
# Initialize a new empty dataframe with the same index
new_metadata = pd.DataFrame(index=metadata.index)

# Now your code works perfectly!
new_metadata["estrogen_receptor_status"] = metadata["estrogen_receptor_status"]

In [13]:

common_ids = genes.index.intersection(new_metadata.index)
X = genes.loc[common_ids]
y = new_metadata.loc[common_ids]

print(f"Aligned: {X.shape[0]} samples, {X.shape[1]} genes")



# sanity check — this should show NO NaNs; if it does, some values
# in ch1_2 weren't exactly "LMP" or "Malignant" (e.g. whitespace)

print(y.value_counts())

# 3. Convert to plain numpy arrays (this is the step that fixes the earlier bug)
X_arr = X.to_numpy(dtype=float)
y_arr = y.to_numpy(dtype=int)

Aligned: 198 samples, 22283 genes
estrogen_receptor_status
1                           134
0                            64
Name: count, dtype: int64


# QuantileNormalizer

In [14]:
class QuantileNormalizer:
    """
    Quantile Normalization for microarray data.

    Transforms each sample to have identical distribution by mapping
    to a reference distribution computed as the mean across all samples.

    Reference: Eq. (1) and (2) in paper
    """

    def __init__(self):
        self.reference_distribution = None

    def fit(self, X):
        """
        Compute reference distribution from training data.

        Args:
            X: Expression matrix (samples x genes)
        """
        X = np.array(X)
        n_samples, n_genes = X.shape

        # Sort each sample and compute mean across samples at each rank
        sorted_data = np.sort(X, axis=1)
        self.reference_distribution = np.mean(sorted_data, axis=0)

        return self

    def transform(self, X):
        """
        Apply quantile normalization.

        Args:
            X: Expression matrix (samples x genes)

        Returns:
            Normalized expression matrix
        """
        X = np.array(X)
        n_samples, n_genes = X.shape
        X_normalized = np.zeros_like(X)

        for i in range(n_samples):
            # Get ranks of original values
            ranks = stats.rankdata(X[i], method='ordinal') - 1
            # Map to reference distribution
            X_normalized[i] = self.reference_distribution[ranks.astype(int)]

        return X_normalized

    def fit_transform(self, X):
        """Fit and transform in one step."""
        self.fit(X)
        return self.transform(X)


# ComBatCorrection

In [15]:
class ComBatCorrection:
    """
    ComBat batch effect correction using empirical Bayes estimation.

    Removes systematic batch effects while preserving biological variation.

    Reference: Eq. (3) and (4) in paper
    """

    def __init__(self):
        self.grand_mean = None
        self.batch_means = None
        self.batch_vars = None
        self.gamma_star = None
        self.delta_star = None

    def fit(self, X, batch_labels):
        """
        Estimate batch effect parameters.

        Args:
            X: Expression matrix (samples x genes)
            batch_labels: Batch assignment for each sample
        """
        X = np.array(X)
        batch_labels = np.array(batch_labels)
        unique_batches = np.unique(batch_labels)

        n_samples, n_genes = X.shape

        # Compute grand mean
        self.grand_mean = np.mean(X, axis=0)

        # Compute batch-specific parameters
        self.batch_means = {}
        self.batch_vars = {}

        for batch in unique_batches:
            mask = batch_labels == batch
            batch_data = X[mask]
            self.batch_means[batch] = np.mean(batch_data, axis=0)
            self.batch_vars[batch] = np.var(batch_data, axis=0) + 1e-10

        # Empirical Bayes estimation of batch effects
        self.gamma_star = {}
        self.delta_star = {}

        for batch in unique_batches:
            # Additive batch effect (standardized)
            self.gamma_star[batch] = self.batch_means[batch] - self.grand_mean
            # Multiplicative batch effect
            pooled_var = np.mean([self.batch_vars[b] for b in unique_batches], axis=0)
            self.delta_star[batch] = np.sqrt(self.batch_vars[batch] / (pooled_var + 1e-10))

        return self

    def transform(self, X, batch_labels):
        """
        Apply batch correction.

        Args:
            X: Expression matrix (samples x genes)
            batch_labels: Batch assignment for each sample

        Returns:
            Batch-corrected expression matrix
        """
        X = np.array(X)
        batch_labels = np.array(batch_labels)
        X_corrected = np.zeros_like(X)

        for i, batch in enumerate(batch_labels):
            if batch in self.gamma_star:
                # Apply correction: (Y - gamma*) / delta* + grand_mean
                X_corrected[i] = (X[i] - self.gamma_star[batch]) / (self.delta_star[batch] + 1e-10) + self.grand_mean
            else:
                X_corrected[i] = X[i]

        return X_corrected

    def fit_transform(self, X, batch_labels):
        """Fit and transform in one step."""
        self.fit(X, batch_labels)
        return self.transform(X, batch_labels)


# KNNImputer

In [16]:
class KNNImputer:
    """
    K-Nearest Neighbors imputation with weighted averaging.

    Reference: Eq. (5) and (6) in paper
    """

    def __init__(self, n_neighbors=10, sigma=1.0):
        """
        Args:
            n_neighbors: Number of nearest neighbors (K)
            sigma: Bandwidth parameter for Gaussian weights
        """
        self.n_neighbors = n_neighbors
        self.sigma = sigma

    def fit_transform(self, X):
        """
        Impute missing values using KNN.

        Args:
            X: Expression matrix with possible NaN values

        Returns:
            Imputed expression matrix
        """
        X = np.array(X, dtype=float)
        n_samples, n_genes = X.shape

        # Find missing values
        missing_mask = np.isnan(X)

        if not np.any(missing_mask):
            return X

        X_imputed = X.copy()

        for i in range(n_samples):
            if not np.any(missing_mask[i]):
                continue

            # Find non-missing genes for this sample
            valid_genes = ~missing_mask[i]
            missing_genes = missing_mask[i]

            # Compute distances to other samples using valid genes
            distances = []
            for j in range(n_samples):
                if i == j or np.any(missing_mask[j, valid_genes]):
                    distances.append(np.inf)
                else:
                    dist = np.sqrt(np.sum((X[i, valid_genes] - X[j, valid_genes])**2))
                    distances.append(dist)

            distances = np.array(distances)

            # Get K nearest neighbors
            neighbor_indices = np.argsort(distances)[:self.n_neighbors]
            neighbor_distances = distances[neighbor_indices]

            # Compute Gaussian weights
            weights = np.exp(-neighbor_distances**2 / (2 * self.sigma**2))
            weights = weights / (np.sum(weights) + 1e-10)

            # Impute missing values
            for gene_idx in np.where(missing_genes)[0]:
                neighbor_values = X[neighbor_indices, gene_idx]
                valid_neighbor_mask = ~np.isnan(neighbor_values)
                if np.any(valid_neighbor_mask):
                    X_imputed[i, gene_idx] = np.sum(
                        weights[valid_neighbor_mask]
                        * neighbor_values[valid_neighbor_mask]
                    ) / np.sum(weights[valid_neighbor_mask])
                else:
                    X_imputed[i, gene_idx] = np.nanmean(X[:, gene_idx])
        return X_imputed


# AttentionFeatureSelector

In [17]:
class AttentionFeatureSelector:
    """
    Attention-weighted feature selection mechanism.

    Computes attention scores that quantify discriminative importance of each gene
    based on statistical embeddings and mutual information.

    Reference: Eq. (7), (8), and (9) in paper
    """

    def __init__(self, embedding_dim=64, mad_threshold_percentile=25):
        """
        Args:
            embedding_dim: Dimension of gene embeddings (d)
            mad_threshold_percentile: Percentile for MAD filtering threshold
        """
        self.embedding_dim = embedding_dim
        self.mad_threshold_percentile = mad_threshold_percentile
        self.attention_scores = None
        self.selected_genes = None
        self.W_e = None
        self.b_e = None
        self.W_h = None
        self.b_h = None
        self.w = None

    def _compute_statistics(self, X, y):
        """
        Compute statistical features for each gene.

        Args:
            X: Expression matrix (samples x genes)
            y: Class labels

        Returns:
            Statistical features (genes x 5): [mean, std, skewness, kurtosis, MI]
        """
        n_genes = X.shape[1]
        stats_features = np.zeros((n_genes, 5))

        for j in range(n_genes):
            gene_expr = X[:, j]
            stats_features[j, 0] = np.mean(gene_expr)  # Mean
            stats_features[j, 1] = np.std(gene_expr)   # Std
            stats_features[j, 2] = stats.skew(gene_expr)  # Skewness
            stats_features[j, 3] = stats.kurtosis(gene_expr)  # Kurtosis

        # Compute mutual information
        mi_scores = mutual_info_classif(X, y, random_state=42)
        stats_features[:, 4] = mi_scores

        # Normalize features
        stats_features = (stats_features - np.mean(stats_features, axis=0)) / (np.std(stats_features, axis=0) + 1e-10)

        return stats_features

    def _compute_embeddings(self, stats_features):
        """
        Compute gene embeddings from statistical features.

        Reference: Eq. (8)

        Args:
            stats_features: Statistical features (genes x 5)

        Returns:
            Gene embeddings (genes x embedding_dim)
        """
        n_genes = stats_features.shape[0]

        # Initialize learnable parameters (using random initialization)
        np.random.seed(42)
        self.W_e = np.random.randn(5, self.embedding_dim) * 0.1
        self.b_e = np.zeros(self.embedding_dim)

        # Compute embeddings: h_j = sigmoid(W_e @ stats + b_e)
        embeddings = 1 / (1 + np.exp(-(stats_features @ self.W_e + self.b_e)))

        return embeddings

    def _compute_attention_scores(self, embeddings):
        """
        Compute attention scores for each gene.

        Reference: Eq. (7)

        Args:
            embeddings: Gene embeddings (genes x embedding_dim)

        Returns:
            Attention scores (genes,)
        """
        n_genes = embeddings.shape[0]

        # Initialize attention parameters
        np.random.seed(43)
        self.W_h = np.random.randn(self.embedding_dim, self.embedding_dim) * 0.1
        self.b_h = np.zeros(self.embedding_dim)
        self.w = np.random.randn(self.embedding_dim) * 0.1

        # Compute attention: a_j = softmax(w^T * tanh(W_h @ h_j + b_h))
        hidden = np.tanh(embeddings @ self.W_h + self.b_h)
        scores = hidden @ self.w

        # Softmax normalization
        exp_scores = np.exp(scores - np.max(scores))
        attention_scores = exp_scores / np.sum(exp_scores)

        return attention_scores

    def fit(self, X, y):
        """
        Compute attention scores and select top genes.

        Args:
            X: Expression matrix (samples x genes)
            y: Class labels
        """
        X = np.array(X)
        y = np.array(y)
        n_genes = X.shape[1]

        # MAD filtering
        mad_values = np.median(np.abs(X - np.median(X, axis=0)), axis=0)
        mad_threshold = np.percentile(mad_values, self.mad_threshold_percentile)
        self.high_mad_genes = mad_values > mad_threshold

        print(f"MAD filtering: {np.sum(self.high_mad_genes)} genes pass threshold")

        # Compute statistics for genes passing MAD filter
        X_filtered = X[:, self.high_mad_genes]
        stats_features = self._compute_statistics(X_filtered, y)

        # Compute embeddings
        embeddings = self._compute_embeddings(stats_features)

        # Compute attention scores
        attention_filtered = self._compute_attention_scores(embeddings)

        # Map back to full gene set
        self.attention_scores = np.zeros(n_genes)
        self.attention_scores[self.high_mad_genes] = attention_filtered

        return self

    def get_top_genes(self, n_genes):
        """
        Get indices of top n genes by attention score.

        Args:
            n_genes: Number of genes to select

        Returns:
            Indices of selected genes
        """
        return np.argsort(self.attention_scores)[-n_genes:]

    def transform(self, X, selected_indices):
        """
        Select genes based on provided indices.

        Args:
            X: Expression matrix
            selected_indices: Indices of genes to keep

        Returns:
            Reduced expression matrix
        """
        return X[:, selected_indices]


# AdaptiveEnsembleClassifier

In [18]:
class AdaptiveEnsembleClassifier(BaseEstimator, ClassifierMixin):
    """
    Adaptive Ensemble Classifier combining GBM, RF, and SVM.

    Features:
    - Gradient Boosting Machine for nonlinear interactions
    - Random Forest for variance reduction
    - Support Vector Machine for high-dimensional effectiveness
    - Adaptive weight optimization using SLSQP
    - Probability calibration using isotonic regression

    Reference: Eq. (10)-(18) in paper
    """

    def __init__(self,
                 gbm_params=None,
                 rf_params=None,
                 svm_params=None,
                 calibrate=True,
                 random_state=42):
        """
        Args:
            gbm_params: Parameters for Gradient Boosting
            rf_params: Parameters for Random Forest
            svm_params: Parameters for SVM
            calibrate: Whether to calibrate probabilities
            random_state: Random seed
        """
        self.gbm_params = gbm_params or {
            'learning_rate': 0.1,
            'max_depth': 6,
            'n_estimators': 200,
            'random_state': random_state
        }
        self.rf_params = rf_params or {
            'n_estimators': 500,
            'max_features': 'sqrt',
            'random_state': random_state
        }
        self.svm_params = svm_params or {
            'C': 1.0,
            'kernel': 'rbf',
            'gamma': 'scale',
            'probability': True,
            'random_state': random_state
        }
        self.calibrate = calibrate
        self.random_state = random_state

        # Initialize classifiers
        self.gbm = GradientBoostingClassifier(**self.gbm_params)
        self.rf = RandomForestClassifier(**self.rf_params)
        self.svm = SVC(**self.svm_params)

        # Ensemble weights (optimized during training)
        self.weights = np.array([1/3, 1/3, 1/3])

        # Calibration models
        self.calibrators = {}

        # Training history
        self.training_history = {'gbm_loss': [], 'rf_oob': [], 'svm_loss': []}

    def fit(self, X, y, X_val=None, y_val=None):
        """
        Train ensemble classifiers and optimize weights.

        Args:
            X: Training features
            y: Training labels
            X_val: Validation features (optional, for weight optimization)
            y_val: Validation labels (optional)
        """
        X = np.array(X)
        y = np.array(y)

        print("Training Gradient Boosting...")
        self.gbm.fit(X, y)

        print("Training Random Forest...")
        self.rf.fit(X, y)

        print("Training SVM...")
        self.svm.fit(X, y)

        # Record training history for GBM
        if hasattr(self.gbm, 'train_score_'):
            self.training_history['gbm_loss'] = self.gbm.train_score_.tolist()

        # Record OOB score for RF
        if hasattr(self.rf, 'oob_score_'):
            self.training_history['rf_oob'] = [self.rf.oob_score_]

        # Optimize weights if validation data provided
        if X_val is not None and y_val is not None:
            self._optimize_weights(X_val, y_val)
        else:
            # Use cross-validation to optimize weights
            self._optimize_weights_cv(X, y)

        # Calibrate probabilities
        if self.calibrate:
            self._fit_calibrators(X, y)

        return self

    def _optimize_weights(self, X_val, y_val):
        """
        Optimize ensemble weights using SLSQP.

        Reference: Eq. (17)
        """
        def cross_entropy_loss(weights):
            """Compute cross-entropy loss for weight combination."""
            weights = weights / np.sum(weights)  # Normalize

            # Get predictions from each classifier
            p_gbm = self.gbm.predict_proba(X_val)
            p_rf = self.rf.predict_proba(X_val)
            p_svm = self.svm.predict_proba(X_val)

            # Weighted combination
            p_ensemble = weights[0] * p_gbm + weights[1] * p_rf + weights[2] * p_svm
            p_ensemble = np.clip(p_ensemble, 1e-10, 1 - 1e-10)

            # Cross-entropy loss
            n_samples = len(y_val)
            loss = 0
            for i in range(n_samples):
                loss -= np.log(p_ensemble[i, y_val[i]])

            return loss / n_samples

        # Constraints: weights sum to 1, all non-negative
        constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
        bounds = [(0, 1), (0, 1), (0, 1)]

        # Optimize
        result = minimize(
            cross_entropy_loss,
            x0=np.array([1/3, 1/3, 1/3]),
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )

        self.weights = result.x / np.sum(result.x)
        print(f"Optimized weights: GBM={self.weights[0]:.3f}, RF={self.weights[1]:.3f}, SVM={self.weights[2]:.3f}")

    def _optimize_weights_cv(self, X, y, cv=3):
        """Optimize weights using cross-validation."""
        kfold = StratifiedKFold(n_splits=cv, shuffle=True, random_state=self.random_state)

        all_preds_gbm = []
        all_preds_rf = []
        all_preds_svm = []
        all_labels = []

        for train_idx, val_idx in kfold.split(X, y):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            # Train temporary classifiers
            gbm_temp = GradientBoostingClassifier(**self.gbm_params)
            rf_temp = RandomForestClassifier(**self.rf_params)
            svm_temp = SVC(**self.svm_params)

            gbm_temp.fit(X_train, y_train)
            rf_temp.fit(X_train, y_train)
            svm_temp.fit(X_train, y_train)

            all_preds_gbm.append(gbm_temp.predict_proba(X_val))
            all_preds_rf.append(rf_temp.predict_proba(X_val))
            all_preds_svm.append(svm_temp.predict_proba(X_val))
            all_labels.append(y_val)

        # Concatenate predictions
        p_gbm = np.vstack(all_preds_gbm)
        p_rf = np.vstack(all_preds_rf)
        p_svm = np.vstack(all_preds_svm)
        y_all = np.concatenate(all_labels)

        def cross_entropy_loss(weights):
            weights = weights / np.sum(weights)
            p_ensemble = weights[0] * p_gbm + weights[1] * p_rf + weights[2] * p_svm
            p_ensemble = np.clip(p_ensemble, 1e-10, 1 - 1e-10)
            loss = 0
            for i in range(len(y_all)):
                loss -= np.log(p_ensemble[i, y_all[i]])
            return loss / len(y_all)

        constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
        bounds = [(0, 1), (0, 1), (0, 1)]

        result = minimize(
            cross_entropy_loss,
            x0=np.array([1/3, 1/3, 1/3]),
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )

        self.weights = result.x / np.sum(result.x)
        print(f"Optimized weights (CV): GBM={self.weights[0]:.3f}, RF={self.weights[1]:.3f}, SVM={self.weights[2]:.3f}")

    def _fit_calibrators(self, X, y):
        """
        Fit isotonic regression calibrators.

        Reference: Eq. (18)
        """
        # Get uncalibrated probabilities
        p_ensemble = self._get_uncalibrated_proba(X)

        n_classes = p_ensemble.shape[1]

        for c in range(n_classes):
            y_binary = (y == c).astype(int)
            calibrator = IsotonicRegression(out_of_bounds='clip')
            calibrator.fit(p_ensemble[:, c], y_binary)
            self.calibrators[c] = calibrator

    def _get_uncalibrated_proba(self, X):
        """Get uncalibrated ensemble probabilities."""
        p_gbm = self.gbm.predict_proba(X)
        p_rf = self.rf.predict_proba(X)
        p_svm = self.svm.predict_proba(X)

        p_ensemble = self.weights[0] * p_gbm + self.weights[1] * p_rf + self.weights[2] * p_svm
        return p_ensemble

    def predict_proba(self, X):
        """
        Predict class probabilities.

        Args:
            X: Features

        Returns:
            Calibrated class probabilities
        """
        p_ensemble = self._get_uncalibrated_proba(X)

        if self.calibrate and self.calibrators:
            p_calibrated = np.zeros_like(p_ensemble)
            for c, calibrator in self.calibrators.items():
                p_calibrated[:, c] = calibrator.predict(p_ensemble[:, c])
            # Normalize
            p_calibrated = p_calibrated / (np.sum(p_calibrated, axis=1, keepdims=True) + 1e-10)
            return p_calibrated

        return p_ensemble

    def predict(self, X):
        """Predict class labels."""
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)

    def get_feature_importance(self):
        """Get aggregated feature importance from ensemble."""
        # GBM importance
        gbm_imp = self.gbm.feature_importances_

        # RF importance
        rf_imp = self.rf.feature_importances_

        # Aggregate with ensemble weights
        importance = self.weights[0] * gbm_imp + self.weights[1] * rf_imp
        importance = importance / np.sum(importance)

        return importance


# MICROAI

In [19]:
class MICROAI:
    """
    MICRO-AI: Complete framework for DNA microarray analysis and disease detection.

    Integrates:
    - Quantile normalization
    - ComBat batch effect correction
    - KNN missing value imputation
    - Attention-weighted feature selection
    - RFECV for optimal gene subset selection
    - Adaptive ensemble classification
    - Probability calibration
    """

    def __init__(self,
                 n_selected_features=150,
                 n_neighbors_impute=10,
                 embedding_dim=64,
                 mad_percentile=25,
                 rfecv_step=0.1,
                 cv_folds=5,
                 calibrate=True,
                 random_state=42):
        """
        Args:
            n_selected_features: Target number of selected genes
            n_neighbors_impute: K for KNN imputation
            embedding_dim: Dimension for attention embeddings
            mad_percentile: Percentile threshold for MAD filtering
            rfecv_step: Step size for RFECV
            cv_folds: Number of cross-validation folds
            calibrate: Whether to calibrate probabilities
            random_state: Random seed for reproducibility
        """
        self.n_selected_features = n_selected_features
        self.n_neighbors_impute = n_neighbors_impute
        self.embedding_dim = embedding_dim
        self.mad_percentile = mad_percentile
        self.rfecv_step = rfecv_step
        self.cv_folds = cv_folds
        self.calibrate = calibrate
        self.random_state = random_state

        # Components (initialized during fit)
        self.quantile_normalizer = QuantileNormalizer()
        self.combat_corrector = None
        self.knn_imputer = KNNImputer(n_neighbors=n_neighbors_impute)
        self.attention_selector = AttentionFeatureSelector(
            embedding_dim=embedding_dim,
            mad_threshold_percentile=mad_percentile
        )
        self.scaler = StandardScaler()
        self.ensemble = None

        # Selected gene indices
        self.selected_genes = None
        self.gene_names = None

        # Label encoder
        self.label_encoder = LabelEncoder()

        # Training info
        self.n_classes = None
        self.training_info = {}

    def preprocess(self, X, batch_labels=None, fit=True):
        """
        Apply preprocessing pipeline.

        Args:
            X: Expression matrix (samples x genes)
            batch_labels: Batch assignments (optional)
            fit: Whether to fit preprocessing components

        Returns:
            Preprocessed expression matrix
        """
        X = np.array(X, dtype=float)

        # Step 1: KNN Imputation (if missing values)
        if np.any(np.isnan(X)):
            print("Imputing missing values...")
            X = self.knn_imputer.fit_transform(X)

        # Step 2: Quantile Normalization
        print("Applying quantile normalization...")
        if fit:
            X = self.quantile_normalizer.fit_transform(X)
        else:
            X = self.quantile_normalizer.transform(X)

        # Step 3: ComBat Batch Correction (if batch labels provided)
        if batch_labels is not None:
            print("Applying ComBat batch correction...")
            if fit:
                self.combat_corrector = ComBatCorrection()
                X = self.combat_corrector.fit_transform(X, batch_labels)
            elif self.combat_corrector is not None:
                X = self.combat_corrector.transform(X, batch_labels)

        return X

    def select_features(self, X, y):
        """
        Apply attention-weighted feature selection with RFECV.

        Args:
            X: Preprocessed expression matrix
            y: Class labels

        Returns:
            Reduced expression matrix, selected gene indices
        """
        print("\nFeature Selection Pipeline")
        print("-" * 40)

        # Step 1: Compute attention scores
        print("Computing attention scores...")
        self.attention_selector.fit(X, y)

        # Step 2: Pre-select top genes based on attention
        n_preselect = min(5000, X.shape[1])
        top_genes = self.attention_selector.get_top_genes(n_preselect)
        X_preselected = X[:, top_genes]

        print(f"Pre-selected {len(top_genes)} genes based on attention scores")

        # Step 3: RFECV for final selection
        print("Running RFECV...")
        base_estimator = RandomForestClassifier(
            n_estimators=100,
            random_state=self.random_state,
            n_jobs=-1
        )

        rfecv = RFECV(
            estimator=base_estimator,
            step=self.rfecv_step,
            cv=StratifiedKFold(self.cv_folds, shuffle=True, random_state=self.random_state),
            scoring='accuracy',
            n_jobs=-1,
            min_features_to_select=min(self.n_selected_features, X_preselected.shape[1])
        )

        rfecv.fit(X_preselected, y)

        # Get selected genes
        rfecv_selected = rfecv.support_
        final_genes_in_preselected = np.where(rfecv_selected)[0]
        self.selected_genes = top_genes[final_genes_in_preselected]

        print(f"RFECV selected {len(self.selected_genes)} genes")
        print(f"Optimal CV score: {rfecv.cv_results_['mean_test_score'].max():.4f}")

        # Store training info
        self.training_info['n_genes_original'] = X.shape[1]
        self.training_info['n_genes_after_attention'] = len(top_genes)
        self.training_info['n_genes_final'] = len(self.selected_genes)
        self.training_info['feature_reduction'] = 100 * (1 - len(self.selected_genes) / X.shape[1])

        return X[:, self.selected_genes], self.selected_genes

    def fit(self, X, y, batch_labels=None, gene_names=None):
        """
        Train the complete MICRO-AI pipeline.

        Args:
            X: Expression matrix (samples x genes)
            y: Class labels
            batch_labels: Batch assignments (optional)
            gene_names: Gene identifiers (optional)
        """
        print("=" * 60)
        print("MICRO-AI Training Pipeline")
        print("=" * 60)

        X = np.array(X, dtype=float)
        self.gene_names = gene_names

        # Encode labels
        y_encoded = self.label_encoder.fit_transform(y)
        self.n_classes = len(np.unique(y_encoded))
        self.classes_ = np.unique(y)



        print(f"\nDataset: {X.shape[0]} samples, {X.shape[1]} genes, {self.n_classes} classes")

        # Preprocessing
        print("\n--- Preprocessing ---")
        X_preprocessed = self.preprocess(X, batch_labels, fit=True)

        # Feature selection
        print("\n--- Feature Selection ---")
        X_selected, _ = self.select_features(X_preprocessed, y_encoded)

        # Standardization
        print("\n--- Standardization ---")
        X_scaled = self.scaler.fit_transform(X_selected)

        # Train ensemble
        print("\n--- Ensemble Training ---")
        self.ensemble = AdaptiveEnsembleClassifier(
            calibrate=self.calibrate,
            random_state=self.random_state
        )
        self.ensemble.fit(X_scaled, y_encoded)

        print("\n" + "=" * 60)
        print("Training Complete")
        print("=" * 60)

        return self

    def predict(self, X, batch_labels=None):
        """
        Predict class labels for new samples.

        Args:
            X: Expression matrix
            batch_labels: Batch assignments (optional)

        Returns:
            Predicted class labels (original encoding)
        """
        X = np.array(X, dtype=float)

        # Preprocessing
        X_preprocessed = self.preprocess(X, batch_labels, fit=False)

        # Select features
        X_selected = X_preprocessed[:, self.selected_genes]

        # Standardization
        X_scaled = self.scaler.transform(X_selected)

        # Predict
        y_pred_encoded = self.ensemble.predict(X_scaled)

        # Decode labels
        y_pred = self.label_encoder.inverse_transform(y_pred_encoded)

        return y_pred

    def predict_proba(self, X, batch_labels=None):
        """
        Predict class probabilities for new samples.

        Args:
            X: Expression matrix
            batch_labels: Batch assignments (optional)

        Returns:
            Class probabilities
        """
        X = np.array(X, dtype=float)

        # Preprocessing
        X_preprocessed = self.preprocess(X, batch_labels, fit=False)

        # Select features
        X_selected = X_preprocessed[:, self.selected_genes]

        # Standardization
        X_scaled = self.scaler.transform(X_selected)

        # Predict probabilities
        proba = self.ensemble.predict_proba(X_scaled)

        return proba

    def get_selected_gene_names(self):
        """Get names of selected genes."""
        if self.gene_names is not None and self.selected_genes is not None:
            return [self.gene_names[i] for i in self.selected_genes]
        return self.selected_genes

    def get_feature_importance(self):
        """Get feature importance for selected genes."""
        if self.ensemble is None:
            return None
        return self.ensemble.get_feature_importance()

    def get_attention_scores(self):
        """Get attention scores for all genes."""
        return self.attention_selector.attention_scores


# Evaluate model

In [20]:
def evaluate_model(y_true, y_pred, y_proba=None, average='binary'):
    """
    Compute comprehensive evaluation metrics.

    Args:
        y_true: True labels
        y_pred: Predicted labels
        y_proba: Predicted probabilities (optional)
        average: Averaging method for multi-class

    Returns:
        Dictionary of metrics
    """
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average=average, zero_division=0),
        'recall': recall_score(y_true, y_pred, average=average, zero_division=0),
        'f1': f1_score(y_true, y_pred, average=average, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred)
    }

    # AUC-ROC (requires probabilities)
    if y_proba is not None:
        n_classes = y_proba.shape[1] if len(y_proba.shape) > 1 else 2
        if n_classes == 2:
            metrics['auc_roc'] = roc_auc_score(y_true, y_proba[:, 1])
        else:
            try:
                metrics['auc_roc'] = roc_auc_score(y_true, y_proba, multi_class='ovr', average=average)
            except:
                metrics['auc_roc'] = None

    # Specificity (for binary)
    cm = confusion_matrix(y_true, y_pred)
    if cm.shape[0] == 2:
        tn, fp, fn, tp = cm.ravel()
        metrics['sensitivity'] = tp / (tp + fn) if (tp + fn) > 0 else 0
        metrics['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0
    else:
        # Multi-class: macro average of specificity
        specificities = []
        for i in range(cm.shape[0]):
            tn = np.sum(cm) - np.sum(cm[i, :]) - np.sum(cm[:, i]) + cm[i, i]
            fp = np.sum(cm[:, i]) - cm[i, i]
            specificities.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
        metrics['specificity'] = np.mean(specificities)
        metrics['sensitivity'] = metrics['recall']

    return metrics


In [21]:
X.shape

(198, 22283)

In [24]:
if __name__ == "__main__":
    # Quick test with synthetic data
    print("Testing MICRO-AI with synthetic data...")

    np.random.seed(42)
    n_samples =X_arr.shape[0]
    n_genes = X_arr.shape[1]
    n_classes = 2

    # Generate synthetic microarray data
    # NEW — replace with this:
    X = np.random.randn(n_samples, n_genes)
    discriminative_genes = np.random.choice(n_genes, 50, replace=False)
    y = np.random.randint(0, n_classes, n_samples)

    effect_size = 1.0
    for gene in discriminative_genes:
        per_sample_effect = effect_size + 0.4 * np.random.randn(n_samples)
        X[:, gene] += y * per_sample_effect

    X += np.random.randn(n_samples, n_genes) * 1.0



    # Train-test split
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # # Create and train model
    # model = MICROAI(
    #     n_selected_features=250,
    #     cv_folds=5,
    #     random_state=42,
    #     n_neighbors_impute=10,
    #     embedding_dim=64,
    #     mad_percentile=15,
    #     rfecv_step=0.05,
    #     calibrate=True)

    # model.fit(X_train, y_train)

    # Predict
    # y_pred = model.predict(X_test)
    # y_proba = model.predict_proba(X_test)

    # Evaluate
    # metrics = evaluate_model(y_test, y_pred, y_proba)

    print("\nTest Results:")
    # print("-" * 40)
    # for metric, value in metrics.items():
    #     if value is not None:
    #         print(f"{metric}: {value:.4f}")

    # print("\nSelected genes:", len(model.selected_genes))
    # print(f"Feature reduction: {model.training_info['feature_reduction']:.1f}%")


Testing MICRO-AI with synthetic data...

Test Results:


In [25]:
demo_x = X_test[25:50:5]
demo_y = y_test[25:50:5]

In [26]:
demo_x_df = pd.DataFrame(demo_x)
demo_y_df = pd.DataFrame(demo_y)
demo_x_df.to_csv("GES7390_demo_x.csv",index=False,
    header=False)
demo_x_df.to_csv("GES7390_demo_x.tsv",sep="\t",index=False,header=False)

demo_y_df.to_csv("GES7390_demo_y.csv",index=False,
    header=False)
demo_y_df.to_csv("GES7390_demo_y.tsv",sep="\t",index=False,header=False)

In [27]:
demo_x_df.shape

(3, 22283)

In [ ]:
joblib.dump(model, "/content/drive/MyDrive/MICRO_AI/GSE7390_model.joblib")


In [ ]:
metadata = {
    'task_type': 'binary_classification',
    'positive_class': int(model.classes_[1]),   # cast numpy scalar -> Python int
    'n_genes_original': int(model.training_info['n_genes_original']),
    'n_genes_final': int(model.training_info['n_genes_final']),
    'selected_genes': model.selected_genes.tolist(),
    'classes': model.classes_.tolist(),
}

with open('/content/drive/MyDrive/MICRO_AI/GSE7390_metadata_01.json', 'w') as f:
    json.dump(metadata, f, indent=2)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# y_true = your actual labels
# y_pred = your model's predictions
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix")
plt.show()

AttributeError: 'numpy.ndarray' object has no attribute 'head'

In [ ]:


GSE7390_DATA = X_train

# Set to "samples_by_features" when:
# rows = samples, columns = probe IDs
#
# Set to "features_by_samples" when:
# rows = probe IDs, columns = samples
GSE7390_ORIENTATION = "samples_by_features"

# If available, supply the EXACT ordered feature IDs used by
# each trained model. These may come from the original training
# pipeline, e.g. list(X_train.columns) BEFORE any reordering.
#
# Leave as None only if the dataset's current order is known
# to be the original training order.
GSE7390_TRAINING_FEATURE_ORDER = None
GSE2034_TRAINING_FEATURE_ORDER = None

# Number of reference samples to export for SHAP.
BACKGROUND_SIZE = 100
RANDOM_SEED = 42

# Export directory
OUTPUT_DIR = Path("microai_shap_assets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 2. VALIDATION AND EXPORT FUNCTION
# ============================================================

def export_shap_assets(
    data,
    model_name,
    orientation="samples_by_features",
    training_feature_order=None,
    background_size=100,
    random_seed=42,
    expected_features=22283,
):
    """
    Export:
      {model_name}_probe_ids.csv
      {model_name}_background.csv

    Background CSV contains numeric expression values only.
    Its columns match the exported probe-ID order exactly.

    Important:
    This function does NOT infer or repair training provenance.
    """

    if not isinstance(data, pd.DataFrame):
        raise TypeError(
            f"{model_name}: expected a pandas DataFrame, "
            "not an array. Recover the corresponding probe IDs first."
        )

    if orientation == "features_by_samples":
        data = data.T.copy()
    elif orientation == "samples_by_features":
        data = data.copy()
    else:
        raise ValueError(
            "orientation must be 'samples_by_features' "
            "or 'features_by_samples'"
        )

    if data.empty:
        raise ValueError(f"{model_name}: dataset is empty.")

    # Feature identifiers must come from the real dataset or
    # the authoritative training pipeline.
    dataset_feature_ids = [str(c) for c in data.columns]

    if any(not x.strip() for x in dataset_feature_ids):
        raise ValueError(f"{model_name}: blank feature IDs found.")

    if len(set(dataset_feature_ids)) != len(dataset_feature_ids):
        raise ValueError(
            f"{model_name}: duplicate feature IDs found. "
            "Resolve them using the original training pipeline."
        )

    if training_feature_order is not None:
        ordered_ids = [str(x) for x in training_feature_order]

        if len(set(ordered_ids)) != len(ordered_ids):
            raise ValueError(
                f"{model_name}: training feature order contains duplicates."
            )

        if set(ordered_ids) != set(dataset_feature_ids):
            missing = sorted(set(ordered_ids) - set(dataset_feature_ids))
            extra = sorted(set(dataset_feature_ids) - set(ordered_ids))

            raise ValueError(
                f"{model_name}: feature IDs do not match.\n"
                f"Missing from dataset: {len(missing)}; "
                f"extra in dataset: {len(extra)}.\n"
                f"Example missing IDs: {missing[:10]}\n"
                f"Example extra IDs: {extra[:10]}"
            )

        # Align using the verified training order.
        data.columns = dataset_feature_ids
        data = data.loc[:, ordered_ids]
        ordered_ids = list(data.columns)

    else:
        ordered_ids = dataset_feature_ids
        warnings.warn(
            f"{model_name}: preserving the dataset's current feature "
            "order. This does NOT prove it matches the model's "
            "original training order.",
            stacklevel=2,
        )

    if expected_features is not None and len(ordered_ids) != expected_features:
        raise ValueError(
            f"{model_name}: expected {expected_features:,} features, "
            f"but found {len(ordered_ids):,}. "
            "Check the dataset, filtering, and training feature schema. "
            "Do not truncate or invent IDs to force a match."
        )

    # Confirm expression values are numeric.
    try:
        numeric_data = data.apply(pd.to_numeric, errors="raise")
    except (ValueError, TypeError) as exc:
        raise ValueError(
            f"{model_name}: non-numeric expression values found."
        ) from exc

    values = numeric_data.to_numpy(dtype=np.float64)

    if not np.isfinite(values).all():
        raise ValueError(
            f"{model_name}: missing or infinite values found. "
            "Apply the verified training-time missing-value policy "
            "or fitted imputer; do not fit a new imputer on this "
            "background dataset."
        )

    if len(numeric_data) < 2:
        raise ValueError(
            f"{model_name}: at least two reference samples are required."
        )

    # Select a reproducible subset of reference samples.
    n_background = min(int(background_size), len(numeric_data))
    if n_background < 1:
        raise ValueError("background_size must be positive.")

    background = numeric_data.sample(
        n=n_background,
        random_state=random_seed,
        replace=False,
    )

    # Export the ordered probe IDs.
    ids_path = OUTPUT_DIR / f"{model_name}_probe_ids.csv"
    pd.DataFrame({"probe_id": ordered_ids}).to_csv(
        ids_path, index=False
    )

    # Export numeric expression values only.
    # No sample ID column, so it cannot be mistaken for a model feature.
    background_path = OUTPUT_DIR / f"{model_name}_background.csv"
    background.to_csv(background_path, index=False, float_format="%.10g")

    # Confirm exported column order.
    exported_background = pd.read_csv(background_path, nrows=1)
    exported_ids = pd.read_csv(ids_path)["probe_id"].astype(str).tolist()

    if exported_background.shape[1] != len(exported_ids):
        raise RuntimeError(
            f"{model_name}: exported background width does not "
            "match the number of probe IDs."
        )

    if list(exported_background.columns.astype(str)) != exported_ids:
        raise RuntimeError(
            f"{model_name}: exported background column order mismatch."
        )

    print(f"\n{model_name}: export successful")
    print(f"  Dataset shape:       {numeric_data.shape}")
    print(f"  Probe IDs:           {ids_path.resolve()}")
    print(f"  Background samples:  {n_background}")
    print(f"  Background file:     {background_path.resolve()}")
    print(f"  Background shape:    {background.shape}")

    return {
        "model": model_name,
        "probe_ids_path": str(ids_path.resolve()),
        "background_path": str(background_path.resolve()),
        "n_features": len(ordered_ids),
        "n_background_samples": n_background,
    }


# ============================================================
# 3. EXPORT BOTH MODELS
# ============================================================

gse7390_assets = export_shap_assets(
    data=GSE7390_DATA,
    model_name="GSE7390",
    orientation=GSE7390_ORIENTATION,
    training_feature_order=GSE7390_TRAINING_FEATURE_ORDER,
    background_size=BACKGROUND_SIZE,
    random_seed=RANDOM_SEED,
)

gse2034_assets = export_shap_assets(
    data=GSE2034_DATA,
    model_name="GSE2034",
    orientation=GSE2034_ORIENTATION,
    training_feature_order=GSE2034_TRAINING_FEATURE_ORDER,
    background_size=BACKGROUND_SIZE,
    random_seed=RANDOM_SEED,
)

print("\nExport summary:")
display(pd.DataFrame([gse7390_assets, gse2034_assets]))
